# Лабораторная работа №2 — Современные оптимизаторы: сравнение и эксперименты

---

## Часть A. Реализация оптимизаторов

### Задание A1–A2. Единый интерфейс и реализация методов
Реализуем базовый класс `Optimizer` с методом `step(theta, grad)` и пять оптимизаторов без использования готовых решений из сторонних библиотек:
- `SGD`
- `Momentum`
- `AdaGrad`
- `RMSProp`
- `Adam`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class Optimizer:
    """Базовый класс оптимизатора с единым интерфейсом step(theta, grad)."""
    def step(self, theta, grad):
        raise NotImplementedError


class SGD(Optimizer):
    def __init__(self, lr=0.01):
        self.lr = lr

    def step(self, theta, grad):
        theta = np.asarray(theta, dtype=float)
        grad = np.asarray(grad, dtype=float)
        return theta - self.lr * grad


class Momentum(Optimizer):
    def __init__(self, lr=0.01, beta=0.9):
        self.lr = lr
        self.beta = beta
        self.v = None

    def step(self, theta, grad):
        theta = np.asarray(theta, dtype=float)
        grad = np.asarray(grad, dtype=float)
        if self.v is None:
            self.v = np.zeros_like(theta)
        self.v = self.beta * self.v + grad
        return theta - self.lr * self.v


class AdaGrad(Optimizer):
    def __init__(self, lr=0.1, eps=1e-8):
        self.lr = lr
        self.eps = eps
        self.s = None

    def step(self, theta, grad):
        theta = np.asarray(theta, dtype=float)
        grad = np.asarray(grad, dtype=float)
        if self.s is None:
            self.s = np.zeros_like(theta)
        self.s = self.s + grad ** 2
        return theta - self.lr * grad / (np.sqrt(self.s) + self.eps)


class RMSProp(Optimizer):
    def __init__(self, lr=0.01, rho=0.9, eps=1e-8):
        self.lr = lr
        self.rho = rho
        self.eps = eps
        self.s = None

    def step(self, theta, grad):
        theta = np.asarray(theta, dtype=float)
        grad = np.asarray(grad, dtype=float)
        if self.s is None:
            self.s = np.zeros_like(theta)
        self.s = self.rho * self.s + (1.0 - self.rho) * (grad ** 2)
        return theta - self.lr * grad / (np.sqrt(self.s) + self.eps)


class Adam(Optimizer):
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = None
        self.v = None
        self.t = 0

    def step(self, theta, grad):
        theta = np.asarray(theta, dtype=float)
        grad = np.asarray(grad, dtype=float)
        if self.m is None:
            self.m = np.zeros_like(theta)
            self.v = np.zeros_like(theta)
        self.t += 1
        self.m = self.beta1 * self.m + (1.0 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1.0 - self.beta2) * (grad ** 2)
        m_hat = self.m / (1.0 - self.beta1 ** self.t)
        v_hat = self.v / (1.0 - self.beta2 ** self.t)
        return theta - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

### Задание A3. Проверка корректности
Проверим все 5 оптимизаторов на функции $f(x) = (x - 3)^2 + 2$ (минимум $x^* = 3$, $f(x^*) = 2$), а также убедимся в сохранении внутреннего состояния и соответствии формулам.

In [ ]:
def f_test(x):
    return (x - 3.0) ** 2 + 2.0

def df_test(x):
    return 2.0 * (x - 3.0)

# Проверка точного соответствия формуле на примере 2 шагов Momentum и Adam
opt_m = Momentum(lr=0.1, beta=0.9)
x_m1 = opt_m.step(np.array([0.0]), np.array([-6.0]))
assert np.allclose(x_m1, [0.6]) and np.allclose(opt_m.v, [-6.0])
x_m2 = opt_m.step(x_m1, np.array([2.0 * (0.6 - 3.0)]))
expected_v2 = 0.9 * (-6.0) + (-4.8)
assert np.allclose(opt_m.v, [expected_v2]) and np.allclose(x_m2, [0.6 - 0.1 * expected_v2])

opt_a = Adam(lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8)
x_a1 = opt_a.step(np.array([0.0]), np.array([-6.0]))
assert opt_a.t == 1 and np.allclose(opt_a.m, [-0.6]) and np.allclose(opt_a.v, [0.036])
print("Проверка формул и сохранения состояния: OK!\n")

# Проверка сходимости всех оптимизаторов к x* = 3.0
test_configs = {
    "SGD": SGD(lr=0.1),
    "Momentum": Momentum(lr=0.05, beta=0.8),
    "AdaGrad": AdaGrad(lr=0.5),
    "RMSProp": RMSProp(lr=0.1, rho=0.9),
    "Adam": Adam(lr=0.2),
}

check_rows = []
for name, opt in test_configs.items():
    x = np.array([0.0])
    for _ in range(100):
        x = opt.step(x, df_test(x))
    check_rows.append({
        "Оптимизатор": name,
        "Найденный x*": float(x[0]),
        "f(x*)": float(f_test(x[0])),
        "|x* - 3.0|": float(abs(x[0] - 3.0))
    })

display(pd.DataFrame(check_rows).round(6))

---
## Часть B. Сравнение на функциях

### Задание B1. Квадратичная функция $f(x, y) = x^2 + 100y^2$
Градиент функции: $\nabla f(x, y) = (2x, 200y)$, минимум в точке $(0, 0)$, $f(0, 0) = 0$.  
Все оптимизаторы стартуют из одной точки $(-8.0, 2.0)$ и выполняют `120` итераций.

In [ ]:
def ravine_f(theta):
    return theta[0] ** 2 + 100.0 * (theta[1] ** 2)

def ravine_grad(theta):
    return np.array([2.0 * theta[0], 200.0 * theta[1]])

def run_2d_optimization(optimizer, start_point, n_iter=120):
    theta = np.array(start_point, dtype=float)
    traj = [theta.copy()]
    losses = [ravine_f(theta)]
    for _ in range(n_iter):
        grad = ravine_grad(theta)
        theta = optimizer.step(theta, grad)
        traj.append(theta.copy())
        losses.append(ravine_f(theta))
    return np.array(traj), np.array(losses)

start_2d = [-8.0, 2.0]
n_iter_2d = 120

optimizers_b1 = {
    "SGD (lr=0.008)": SGD(lr=0.008),
    "Momentum (lr=0.005, β=0.8)": Momentum(lr=0.005, beta=0.8),
    "AdaGrad (lr=1.0)": AdaGrad(lr=1.0),
    "RMSProp (lr=0.1, ρ=0.9)": RMSProp(lr=0.1, rho=0.9),
    "Adam (lr=0.2)": Adam(lr=0.2),
}

results_b1 = {}
summary_b1 = []
for label, opt in optimizers_b1.items():
    traj, losses = run_2d_optimization(opt, start_2d, n_iter_2d)
    results_b1[label] = (traj, losses)
    summary_b1.append({
        "Метод": label,
        "Финальный x": traj[-1, 0],
        "Финальный y": traj[-1, 1],
        "Финальное f(x, y)": losses[-1],
        "Итераций": n_iter_2d
    })

display(pd.DataFrame(summary_b1).round(6))

# Графики сходимости и траекторий на линиях уровня
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for label, (traj, losses) in results_b1.items():
    axes[0].plot(losses, lw=2, label=label)
axes[0].set_yscale("log")
axes[0].set_xlabel("Итерация")
axes[0].set_ylabel("f(x, y), лог. шкала")
axes[0].set_title("Сходимость на функции f(x, y) = x² + 100y²")
axes[0].grid(True, which="both", linestyle="--", alpha=0.5)
axes[0].legend()

xg = np.linspace(-9, 3, 300)
yg = np.linspace(-2.5, 2.5, 300)
Xg, Yg = np.meshgrid(xg, yg)
Zg = Xg ** 2 + 100.0 * (Yg ** 2)
axes[1].contour(Xg, Yg, Zg, levels=np.logspace(-1, 2.8, 18), cmap="viridis", alpha=0.55)
for label, (traj, _) in results_b1.items():
    axes[1].plot(traj[:, 0], traj[:, 1], lw=1.8, marker=".", markersize=4, label=label)
axes[1].scatter([0], [0], color="gold", edgecolor="black", marker="*", s=180, zorder=6, label="Минимум (0, 0)")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Траектории оптимизации на линиях уровня")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend(loc="upper right", fontsize=8.5)

plt.tight_layout()
plt.show()

**Наблюдения по заданию B1:**
- **SGD** вынужден использовать маленький шаг `lr=0.008` (иначе разойдётся по крутой оси $y$), поэтому вдоль пологой оси $x$ он движется очень медленно.
- **Momentum** гасит поперечные колебания за счёт накопления импульса вдоль оси $x$ и быстрее продвигается к минимуму.
- **AdaGrad, RMSProp и Adam** адаптируют шаг отдельно для каждой координаты: уменьшают его по оси $y$ (где градиент велик) и увеличивают по оси $x$ (где градиент мал). В результате **Adam** и **RMSProp** выходят в окрестность $(0, 0)$ значительно быстрее обычного SGD. **AdaGrad** в начале быстро спускается, но из-за монотонного накопления $s_t$ к концу притормаживает.

### Задание B2. Влияние гиперпараметров
Исследуем влияние:
1. Параметра $\beta \in \{0.0, 0.5, 0.85, 0.95\}$ для **Momentum** (при фиксированном `lr=0.005`).
2. Параметра $\rho \in \{0.5, 0.9, 0.99\}$ для **RMSProp** (при фиксированном `lr=0.1`).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for beta in [0.0, 0.5, 0.85, 0.95]:
    _, losses = run_2d_optimization(Momentum(lr=0.005, beta=beta), start_2d, n_iter=120)
    axes[0].plot(losses, lw=2, label=f'β = {beta}')
axes[0].set_yscale('log')
axes[0].set_xlabel('Итерация')
axes[0].set_ylabel('f(x, y), лог. шкала')
axes[0].set_title('Momentum: влияние β (lr = 0.005)')
axes[0].grid(True, which='both', linestyle='--', alpha=0.5)
axes[0].legend()

for rho in [0.5, 0.9, 0.99]:
    _, losses = run_2d_optimization(RMSProp(lr=0.1, rho=rho), start_2d, n_iter=120)
    axes[1].plot(losses, lw=2, label=f'ρ = {rho}')
axes[1].set_yscale('log')
axes[1].set_xlabel('Итерация')
axes[1].set_ylabel('f(x, y), лог. шкала')
axes[1].set_title('RMSProp: влияние ρ (lr = 0.1)')
axes[1].grid(True, which='both', linestyle='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

**Вывод по заданию B2:**
- В **Momentum** при $\beta = 0$ метод вырождается в обычный SGD. С ростом $\beta$ ($0.5 \to 0.85$) накопленный импульс ускоряет движение вдоль оврага и ускоряет сходимость. При слишком большом $\beta = 0.95$ инерция становится чрезмерной, возникают заметные осцилляции и «перелёты» через минимум.
- В **RMSProp** параметр $\rho$ задаёт длину «памяти» скользящего среднего квадратов градиента: при $\rho = 0.9$ адаптация шага сбалансирована и сходится быстро; при слишком большом $\rho = 0.99$ оценка $s_t$ меняется медленно и подстраивается с задержкой, а при малом $\rho = 0.5$ слишком сильно реагирует на отдельные шаги.

---
## Часть C. Оптимизаторы в обучении модели (линейная регрессия)

### Задание C1. Подготовка данных
Генерируем $n = 200$ объектов ($y = 2x + 1 + \varepsilon$, `seed = 42`) и делим выборку на обучающую (80%) и тестовую (20%).

In [ ]:
rng = np.random.default_rng(42)
n_samples = 200
X_all = rng.uniform(-5.0, 5.0, size=n_samples)
y_all = 2.0 * X_all + 1.0 + rng.normal(0.0, 1.0, size=n_samples)

perm = rng.permutation(n_samples)
split_idx = int(0.8 * n_samples)
train_idx, test_idx = perm[:split_idx], perm[split_idx:]

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]

print(f"Размер обучающей выборки: {len(X_train)}, тестовой: {len(X_test)}")

### Задание C2–C3. Обучение mini-batch оптимизаторами и сравнение результатов
Используем `batch_size = 32`, `epochs = 50`, одинаковую начальную точку $\theta_0 = (w_0, b_0) = (0.0, 0.0)$ и фиксированный `seed` для перемешивания батчей.

In [ ]:
def mse_loss(theta, x, y):
    w, b = theta
    return float(np.mean((w * x + b - y) ** 2))

def mse_grad(theta, x, y):
    w, b = theta
    err = (w * x + b) - y
    dw = (2.0 / len(x)) * np.sum(err * x)
    db = (2.0 / len(x)) * np.sum(err)
    return np.array([dw, db])

def train_linear_regression(optimizer, X_tr, y_tr, X_te, y_te, epochs=50, batch_size=32, seed=42):
    batch_rng = np.random.default_rng(seed)
    theta = np.array([0.0, 0.0], dtype=float)
    train_history = [mse_loss(theta, X_tr, y_tr)]
    grad_evals = 0

    for _ in range(epochs):
        indices = batch_rng.permutation(len(X_tr))
        for start in range(0, len(X_tr), batch_size):
            batch_idx = indices[start:start + batch_size]
            grad = mse_grad(theta, X_tr[batch_idx], y_tr[batch_idx])
            theta = optimizer.step(theta, grad)
            grad_evals += 1
        train_history.append(mse_loss(theta, X_tr, y_tr))

    return {
        "theta": theta,
        "train_mse": mse_loss(theta, X_tr, y_tr),
        "test_mse": mse_loss(theta, X_te, y_te),
        "history": train_history,
        "grad_evals": grad_evals
    }

ml_optimizers = {
    "SGD (lr=0.02)": SGD(lr=0.02),
    "Momentum (lr=0.01, β=0.9)": Momentum(lr=0.01, beta=0.9),
    "AdaGrad (lr=0.3)": AdaGrad(lr=0.3),
    "RMSProp (lr=0.05, ρ=0.9)": RMSProp(lr=0.05, rho=0.9),
    "Adam (lr=0.1)": Adam(lr=0.1),
}

ml_results = {}
ml_table = []
for name, opt in ml_optimizers.items():
    res = train_linear_regression(opt, X_train, y_train, X_test, y_test, epochs=50, batch_size=32, seed=42)
    ml_results[name] = res
    ml_table.append({
        "Оптимизатор": name,
        "w": res["theta"][0],
        "b": res["theta"][1],
        "Train MSE": res["train_mse"],
        "Test MSE": res["test_mse"],
        "Шагов (вычисл. градиента)": res["grad_evals"]
    })

display(pd.DataFrame(ml_table).round(5))

plt.figure(figsize=(10, 5))
for name, res in ml_results.items():
    plt.plot(res["history"], lw=2, label=name)
plt.yscale("log")
plt.xlabel("Эпоха")
plt.ylabel("Train MSE (лог. шкала)")
plt.title("Сходимость оптимизаторов при обучении линейной регрессии (batch_size=32)")
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

#### Ответы на вопросы (Часть C)
1. **Какой оптимизатор достиг наименьшей ошибки?**  
   На выпуклой задаче линейной регрессии при подобранном шаге все методы сходятся к одному глобальному минимуму (Train MSE $\approx 0.90$, Test MSE $\approx 1.01$), минимальная разница в тысячных долях определяется шумом батчей и шагом в конце обучения.
2. **Какой метод сходился быстрее?**  
   Быстрее всех вышли на плато **Momentum** и **Adam** (уже за первые 5–10 эпох), затем **RMSProp** и **AdaGrad**.
3. **Насколько результаты зависят от learning rate?**  
   Очень сильно: при слишком малом `lr` оптимизатор не успевает дойти до минимума за фиксированное число эпох, а при слишком большом — колеблется вокруг минимума или расходится.
4. **Почему нельзя утверждать, что один оптимизатор всегда лучше остальных?**  
   Эффективность зависит от геометрии функции потерь, зашумлённости градиентов, размера батча и настройки гиперпараметров (включая расписание `lr`).
5. **Чем сравнение по эпохам отличается от сравнения по количеству вычислений градиента?**  
   Одна эпоха при `batch_size = 32` на 160 объектах делает 5 вычислений градиента, а полный GD за одну эпоху делает 1 вычисление градиента по всей выборке. Поэтому при разном размере батча одинаковое число эпох соответствует разному числу обновлений параметров и вычислений градиента.

---
## Бонусная часть (до +2 баллов)


### Бонус 2. AdamW vs Adam с L2-регуляризацией (до +2 баллов)
В **Adam + L2** слагаемое $\lambda \theta_t$ добавляется к градиенту $g_t$, поэтому попадает под адаптивное масштабирование $1 / \sqrt{\hat{v}_t}$. В **AdamW** (decoupled weight decay) затухание весов применяется напрямую к $\theta_t$: $\theta_{t+1} = \theta_t - \eta \lambda \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \varepsilon}$.

In [ ]:
class AdamW(Optimizer):
    def __init__(self, lr=0.05, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.1):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.m = None
        self.v = None
        self.t = 0

    def step(self, theta, grad):
        theta = np.asarray(theta, dtype=float)
        grad = np.asarray(grad, dtype=float)
        if self.m is None:
            self.m = np.zeros_like(theta)
            self.v = np.zeros_like(theta)
        self.t += 1
        self.m = self.beta1 * self.m + (1.0 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1.0 - self.beta2) * (grad ** 2)
        m_hat = self.m / (1.0 - self.beta1 ** self.t)
        v_hat = self.v / (1.0 - self.beta2 ** self.t)
        return theta * (1.0 - self.lr * self.weight_decay) - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


wd = 0.2
epochs_wd = 60
batch_size_wd = 32

# Adam + L2
opt_adam_l2 = Adam(lr=0.05)
theta_l2 = np.array([0.0, 0.0])
rng_l2 = np.random.default_rng(42)
norms_l2, loss_l2 = [], []
for _ in range(epochs_wd):
    idx = rng_l2.permutation(len(X_train))
    for s in range(0, len(X_train), batch_size_wd):
        b_idx = idx[s:s + batch_size_wd]
        g = mse_grad(theta_l2, X_train[b_idx], y_train[b_idx]) + wd * theta_l2
        theta_l2 = opt_adam_l2.step(theta_l2, g)
    norms_l2.append(np.linalg.norm(theta_l2))
    loss_l2.append(mse_loss(theta_l2, X_test, y_test))

# AdamW
opt_adamw = AdamW(lr=0.05, weight_decay=wd)
theta_w = np.array([0.0, 0.0])
rng_w = np.random.default_rng(42)
norms_w, loss_w = [], []
for _ in range(epochs_wd):
    idx = rng_w.permutation(len(X_train))
    for s in range(0, len(X_train), batch_size_wd):
        b_idx = idx[s:s + batch_size_wd]
        g = mse_grad(theta_w, X_train[b_idx], y_train[b_idx])
        theta_w = opt_adamw.step(theta_w, g)
    norms_w.append(np.linalg.norm(theta_w))
    loss_w.append(mse_loss(theta_w, X_test, y_test))

print(f"Adam + L2 (λ={wd}): w={theta_l2[0]:.4f}, b={theta_l2[1]:.4f}, ||θ||={norms_l2[-1]:.4f}, Test MSE={loss_l2[-1]:.4f}")
print(f"AdamW     (λ={wd}): w={theta_w[0]:.4f}, b={theta_w[1]:.4f}, ||θ||={norms_w[-1]:.4f}, Test MSE={loss_w[-1]:.4f}")

plt.figure(figsize=(8, 4.5))
plt.plot(norms_l2, lw=2, label='Adam + L2')
plt.plot(norms_w, lw=2, label='AdamW')
plt.xlabel('Эпоха')
plt.ylabel('Норма вектора весов ||θ||')
plt.title(f'Сравнение сжатия весов: Adam + L2 vs AdamW (λ = {wd})')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

**Вывод по Бонусу 2:** в `Adam + L2` регуляризационный член $\lambda \theta_t$ делится на $\sqrt{\hat{v}_t}$, поэтому параметры с большими градиентами (как `w`) сжимаются слабее, чем в `AdamW`, где weight decay действует равномерно и независимо от адаптивного масштаба градиента.

---
## Общий вывод

1. **SGD и Momentum:** обычный SGD чувствителен к плохо обусловленной геометрии функции потерь (вынужден использовать малый `lr`). Momentum сглаживает колебания и ускоряет движение вдоль пологих направлений.
2. **Адаптивные методы (AdaGrad, RMSProp, Adam):** подстраивают эффективный шаг отдельно для каждого параметра. В отличие от AdaGrad, где шаг со временем необратимо затухает, RMSProp и Adam используют экспоненциальное скользящее среднее квадратов градиентов, сохраняя скорость сходимости.
3. **Корректность сравнения:** оптимизаторы необходимо сравнивать при одинаковой стартовой точке, одинаковых данных, фиксированном `random seed` и с учётом числа вычислений градиента, а не только по числу эпох.